Imports

In [5]:
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm
from pathlib import Path

Paths and Directories

In [6]:
DATASET_DIR = Path(os.environ["MHCD2022_PATH"])

Anno_Path = DATASET_DIR / "Annotations"

Labels_Dir = DATASET_DIR.parent / "MHCD2022_YOLO_Labels"
Labels_Dir.mkdir(exist_ok=True)

Catagories

In [7]:
Class_MAP = {
    "person": 0,
    "military vehicle": 1,
    "tank": 2,
    "aeroplane":3,
    "warship": 4   
}

In [8]:
for xml_file in tqdm(os.listdir(Anno_Path)):
    if not xml_file.endswith(".xml"):
        continue

    xml_path = os.path.join(Anno_Path, xml_file)
    
    tree = ET.parse(xml_path)
    root = tree.getroot()

    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)

    yolo_lines = []

    for obj in root.findall("object"):
        
        class_name = obj.find("name").text
        if class_name not in Class_MAP:
            continue

        class_id = Class_MAP[class_name]

        bndbox = obj.find("bndbox")

        xmin = int(bndbox.find("xmin").text)
        ymin = int(bndbox.find("ymin").text)
        xmax = int(bndbox.find("xmax").text)
        ymax = int(bndbox.find("ymax").text)

        x_center = ((xmin + xmax) / 2)/ width
        y_center = ((ymin + ymax) / 2 )/ height

        bbox_width = (xmax - xmin) / width
        bbox_height = (ymax - ymin) / height

        yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}"
        yolo_lines.append(yolo_line)


    txt_name = os.path.splitext(xml_file)[0] + ".txt"

    with open(os.path.join(Labels_Dir, txt_name), "w") as f:
        f.write("\n".join(yolo_lines))

print("YOLO label conversion completed!")            

100%|██████████| 3000/3000 [00:00<00:00, 4272.36it/s]

YOLO label conversion completed!
